# Comprehensiv Climate Risk Mapper — Example Usage

This notebook walks through scoring a small set of fictional villages and shows how a field team would translate the output into a screening priority list. The five villages span the scoring range — from low climate-driven risk to high — so the differences in output are visible.

If you're using this library in your own work, the workflow is the same: load your inputs, score them, sort by composite, decide where to send screening visits first.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
from climate_risk.inputs import LocationInputs
from climate_risk.scoring import composite_score
from climate_risk.outputs import format_summary, priority_order, screening_modules_to_prioritise

## Load the sample data

Five fictional villages. The CSV columns map directly to `LocationInputs` fields.

In [ ]:
df = pd.read_csv('../data/sample_villages.csv')
df

## Build LocationInputs objects from each row

In [ ]:
villages = []
for _, row in df.iterrows():
    villages.append(LocationInputs(
        location_id=row['location_id'],
        name=row['name'],
        aqi=row['aqi'],
        temperature_c=row['temperature_c'],
        humidity_percent=row['humidity_percent'],
        rainfall_7day_mm=row['rainfall_7day_mm'],
        rainfall_30day_mm=row['rainfall_30day_mm'],
        standing_water=bool(row['standing_water']),
        drainage_quality=row['drainage_quality'],
        population_under_5=row['population_under_5'],
        population_5_to_18=row['population_5_to_18'],
    ))

for v in villages:
    print(f"{v.name}: {v.population_under_5} children under 5")

## Score each village and print summaries

In [ ]:
for v in villages:
    result = composite_score(v)
    print(format_summary(v, result))

## Produce a screening priority list — highest risk first

This is the output a field team would actually use to plan a week's visits.

In [ ]:
ranked = priority_order(villages)

print(f"{'Rank':<6}{'Village':<15}{'Score':<8}{'Children U5':<12}")
print('-' * 45)
for i, (loc, result) in enumerate(ranked, 1):
    print(f"{i:<6}{loc.name:<15}{result['composite']:<8.1f}{loc.population_under_5:<12}")

## What modules to prioritise at each location

The sub-scores translate into specific screening priorities. A village high on heat-risk needs different attention from a village high on water-borne risk.

In [ ]:
for loc, result in ranked:
    print(f"\n{loc.name} (composite {result['composite']:.1f}):")
    for module in screening_modules_to_prioritise(result):
        print(f"  - {module}")

## What this output is and isn't

Devripura ranks highest. That means *given the environmental conditions*, children there face the highest risk profile across the four axes we measure. It does not mean Devripura's children are currently sick. It means: if a field team has time to visit two of these five villages this week, Devripura should be one of them, and the team should arrive prepared for diarrhoeal disease, vector-borne disease, and heat-related presentations specifically.

Eklavyapur ranks lowest. That doesn't mean it's safe to skip — it means the climate isn't actively pushing risk up right now, so the screening protocol there can follow routine prioritisation rather than environment-driven prioritisation.

A village absent from this list — because no one entered data for it — is not at zero risk. It's at unknown risk.